# 05 — Pilot Model Training and Robustness Evaluation

This notebook evaluates a baseline text classifier using the frozen
weak-labelled pilot and the controlled noise/preprocessing matrix generated in
Notebook 04.

The modelling objective is to test the feasibility of the experimental
pipeline and to measure how classifier behaviour changes across clean, noisy,
and cleaned-noisy text conditions.

## Pilot modelling constraints

The pilot contains:

- 100 unique source conversations;
- 6 weak-label-positive source conversations;
- 94 weak-label-negative source conversations.

Because the positive class is very small, a conventional single
train/validation/test split would produce highly unstable evaluation subsets.

This notebook therefore uses **3-fold stratified cross-validation at the
source-conversation level**.

Each source conversation belongs to exactly one fold, and all derived variants
of that source remain in the same fold. This prevents leakage between training
and evaluation data.

With six positive sources, the three folds are constructed so that each
held-out fold contains approximately two positive examples.

## Training strategy

The classifier is trained only on the original clean text from the training
sources:

- `noise_condition = clean`
- `preprocess_config = none`

The baseline model is:

- TF-IDF text representation;
- linear Support Vector Machine (SVM);
- class weighting to reduce the effect of the severe class imbalance.

The model is not trained on noisy or preprocessed variants.

This is intentional. Training on all derived variants would expose the model
to the experimental perturbations and would make subsequent robustness
comparisons difficult to interpret.

## Robustness evaluation

For each cross-validation fold:

1. the model is trained using only clean training-source responses;
2. predictions are generated for the held-out source conversations;
3. the same held-out sources are evaluated under every combination of:
   - noise condition;
   - preprocessing configuration.

Each held-out source therefore receives predictions for all 32 derived text
variants while remaining completely unseen during model fitting.

Across the three folds, this produces out-of-fold predictions for every source
conversation under every experimental condition.

The main evaluation metrics will include:

- precision;
- recall;
- F1 score;
- balanced accuracy;
- confusion-matrix counts.

Because only six positive source examples are available, these metrics are
treated as **pilot feasibility and robustness diagnostics**, not as reliable
estimates of generalisable classifier performance.

No hyperparameter optimisation is performed on this pilot. Fixed baseline
settings are used to avoid overfitting methodological choices to the very
small positive class.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/workspaces/irp-disempowerment-nlp")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RANDOM_SEED = 42

matrix_path = (
    PROJECT_ROOT
    / "data"
    / "samples"
    / "lmsys_noise_preprocess_v1_matrix_3200.csv"
)

print("Project root:", PROJECT_ROOT)
print("Matrix path:", matrix_path)
print("Matrix exists:", matrix_path.exists())

In [ ]:
experiment_df = pd.read_csv(matrix_path)

print("Rows:", len(experiment_df))
print(
    "Unique source conversations:",
    experiment_df["source_index"].nunique(),
)
print(
    "Noise conditions:",
    experiment_df["noise_condition"].nunique(),
)
print(
    "Preprocessing configs:",
    experiment_df["preprocess_config"].nunique(),
)
print(
    "Noise versions:",
    experiment_df["noise_version"].unique().tolist(),
)
print(
    "Preprocessing versions:",
    experiment_df["preprocess_version"].unique().tolist(),
)
print(
    "Weak-label versions:",
    experiment_df["weak_label_version"].unique().tolist(),
)

In [ ]:
assert len(experiment_df) == 3200

assert (
    experiment_df["source_index"].nunique()
    == 100
)

assert (
    experiment_df["noise_condition"].nunique()
    == 8
)

assert (
    experiment_df["preprocess_config"].nunique()
    == 4
)

assert (
    experiment_df["noise_version"]
    .eq("noise_v1")
    .all()
)

assert (
    experiment_df["preprocess_version"]
    .eq("preprocess_v1")
    .all()
)

assert (
    experiment_df["weak_label_version"]
    .eq("disempowerment_weak_v5")
    .all()
)

assert (
    experiment_df["assistant_text_processed"]
    .notna()
    .all()
)


# Every source must have all 32 derived variants.
variants_per_source = (
    experiment_df
    .groupby(
        ["source_index", "pair_index"]
    )
    .size()
)

assert (
    variants_per_source == 32
).all()


# Weak labels must be invariant within source.
source_label_counts = (
    experiment_df
    .groupby(
        ["source_index", "pair_index"]
    )["weak_label_any"]
    .nunique(dropna=False)
)

assert (
    source_label_counts == 1
).all()


print("Frozen modelling matrix integrity checks passed.")

In [ ]:
# ------------------------------------------------------------
# Construct source-level modelling table
# ------------------------------------------------------------

clean_none_df = experiment_df[
    (experiment_df["noise_condition"] == "clean")
    & (experiment_df["preprocess_config"] == "none")
].copy()

source_df = (
    clean_none_df[
        [
            "source_index",
            "pair_index",
            "user_text",
            "assistant_text_processed",
            "weak_label_any",
            "directive_advice",
            "sycophantic_validation",
            "overconfident_judgement",
        ]
    ]
    .drop_duplicates(
        [
            "source_index",
            "pair_index",
        ]
    )
    .sort_values(
        [
            "source_index",
            "pair_index",
        ]
    )
    .reset_index(drop=True)
)

source_df = source_df.rename(
    columns={
        "assistant_text_processed": "model_text",
        "weak_label_any": "target",
    }
)

print("Source-level rows:", len(source_df))
print(
    "Unique sources:",
    source_df["source_index"].nunique(),
)

In [ ]:
# ------------------------------------------------------------
# Verify pilot class distribution
# ------------------------------------------------------------

class_counts = (
    source_df["target"]
    .value_counts()
    .sort_index()
)

positive_count = int(
    source_df["target"].sum()
)

negative_count = int(
    len(source_df) - positive_count
)

positive_pct = (
    positive_count
    / len(source_df)
    * 100
)

negative_pct = (
    negative_count
    / len(source_df)
    * 100
)

print("Class distribution:")
print("Negative:", negative_count)
print("Positive:", positive_count)
print(
    "Positive percentage:",
    round(positive_pct, 1),
)
print(
    "Negative percentage:",
    round(negative_pct, 1),
)


assert len(source_df) == 100
assert source_df["source_index"].nunique() == 100
assert positive_count == 6
assert negative_count == 94

assert (
    source_df["model_text"]
    .notna()
    .all()
)

print(
    "\nSource-level modelling table integrity checks passed."
)

In [ ]:
from sklearn.model_selection import StratifiedKFold


# ------------------------------------------------------------
# Create deterministic 3-fold stratified source-level splits
# ------------------------------------------------------------

N_SPLITS = 3

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_SEED,
)

source_df["fold"] = -1

for fold_id, (_, test_idx) in enumerate(
    skf.split(
        source_df,
        source_df["target"],
    )
):
    source_df.loc[
        test_idx,
        "fold",
    ] = fold_id


assert (
    source_df["fold"] >= 0
).all()

print("Assigned folds:", sorted(source_df["fold"].unique()))

In [ ]:
# ------------------------------------------------------------
# Verify fold balance and source isolation
# ------------------------------------------------------------

fold_summary = (
    source_df
    .groupby("fold")
    .agg(
        total_sources=("source_index", "size"),
        positive_sources=("target", "sum"),
    )
)

fold_summary["negative_sources"] = (
    fold_summary["total_sources"]
    - fold_summary["positive_sources"]
)

fold_summary["positive_pct"] = (
    fold_summary["positive_sources"]
    / fold_summary["total_sources"]
    * 100
).round(1)


# Each fold must contain exactly two of the six positives.
assert (
    fold_summary["positive_sources"] == 2
).all()


# Every source must appear in exactly one fold.
assert (
    source_df[
        ["source_index", "pair_index"]
    ]
    .duplicated()
    .sum()
    == 0
)

assert (
    source_df["fold"]
    .value_counts()
    .sum()
    == 100
)


# No source may occur in more than one fold.
folds_per_source = (
    source_df
    .groupby(
        ["source_index", "pair_index"]
    )["fold"]
    .nunique()
)

assert (
    folds_per_source == 1
).all()


print("Fold summary:")
fold_summary

print("\n3-fold source-level split integrity checks passed.")

In [ ]:
# ------------------------------------------------------------
# Propagate source-level fold assignments to all derived variants
# ------------------------------------------------------------

fold_map = source_df[
    [
        "source_index",
        "pair_index",
        "fold",
    ]
].copy()

model_matrix_df = experiment_df.merge(
    fold_map,
    on=[
        "source_index",
        "pair_index",
    ],
    how="left",
    validate="many_to_one",
)


print("Model-matrix rows:", len(model_matrix_df))
print(
    "Rows with missing fold:",
    int(model_matrix_df["fold"].isna().sum()),
)
print(
    "Assigned folds:",
    sorted(model_matrix_df["fold"].unique()),
)

In [ ]:
# ------------------------------------------------------------
# Verify no source leakage across folds
# ------------------------------------------------------------

assert len(model_matrix_df) == 3200

assert (
    model_matrix_df["fold"]
    .notna()
    .all()
)


# Every source must still have exactly 32 variants.
variants_per_source = (
    model_matrix_df
    .groupby(
        [
            "source_index",
            "pair_index",
        ]
    )
    .size()
)

assert (
    variants_per_source == 32
).all()


# Every derived variant of a source must share the same fold.
folds_per_source = (
    model_matrix_df
    .groupby(
        [
            "source_index",
            "pair_index",
        ]
    )["fold"]
    .nunique()
)

assert (
    folds_per_source == 1
).all()


# Every experimental cell should contain all 100 sources.
cell_source_counts = (
    model_matrix_df
    .groupby(
        [
            "noise_condition",
            "preprocess_config",
        ]
    )[
        [
            "source_index",
            "pair_index",
        ]
    ]
    .apply(
        lambda frame:
        frame.drop_duplicates().shape[0]
    )
)

assert (
    cell_source_counts == 100
).all()


print(
    "Fold propagation and source-leakage checks passed."
)

## Baseline model specification

The pilot classifier uses a fixed TF-IDF + linear Support Vector Machine
baseline.

The model is deliberately simple and transparent so that subsequent changes in
performance can be attributed primarily to noise and preprocessing conditions
rather than to extensive model tuning.

### TF-IDF representation

The vectorizer uses:

- word-level features;
- unigrams and bigrams: `ngram_range=(1, 2)`;
- lowercase conversion: `lowercase=True`;
- L2 normalization: `norm="l2"`;
- sublinear term-frequency scaling: `sublinear_tf=True`;
- `min_df=1`;
- no fixed maximum feature limit.

Unigrams capture individual lexical cues, while bigrams allow the baseline to
represent short constructions such as:

- `you should`
- `you must`
- `not appropriate`
- `need to`

The TF-IDF vocabulary is fitted **only on the clean training responses within
each fold**. Held-out text is transformed using that fitted training
vocabulary. This prevents vocabulary leakage from the evaluation sources.

### Linear SVM classifier

The classifier uses scikit-learn's `LinearSVC` with:

- `C=1.0`;
- `class_weight="balanced"`;
- deterministic random seed where applicable.

Class weighting is used because the source-level pilot contains 6 positive and
94 negative examples.

No hyperparameter optimisation or model selection is performed on this pilot.
The configuration is fixed before evaluation.

### Cross-validation training rule

For each of the three folds:

1. sources assigned to the other two folds form the training set;
2. only each training source's `clean + none` response is used for fitting;
3. TF-IDF is fitted on those training responses;
4. the linear SVM is fitted using their frozen weak labels;
5. the held-out fold is evaluated across all 32 noise/preprocessing variants.

The TF-IDF vectorizer and classifier are refitted independently inside each
fold.

This design produces out-of-fold predictions for every source without fitting
on any derived variant belonging to that held-out source.

Because each training fold contains only four positive examples, results are
interpreted as pilot feasibility and robustness diagnostics rather than stable
estimates of generalisable performance.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


# ------------------------------------------------------------
# Fixed baseline model
# ------------------------------------------------------------

def build_baseline_model():
    return Pipeline(
        steps=[
            (
                "tfidf",
                TfidfVectorizer(
                    analyzer="word",
                    ngram_range=(1, 2),
                    lowercase=True,
                    norm="l2",
                    sublinear_tf=True,
                    min_df=1,
                ),
            ),
            (
                "svm",
                LinearSVC(
                    C=1.0,
                    class_weight="balanced",
                    random_state=RANDOM_SEED,
                ),
            ),
        ]
    )


baseline_model = build_baseline_model()

print("Baseline pipeline created.")
print(baseline_model)

In [ ]:
# ------------------------------------------------------------
# Single-fold smoke test
# ------------------------------------------------------------

SMOKE_TEST_FOLD = 0

train_df = source_df[
    source_df["fold"] != SMOKE_TEST_FOLD
].copy()

test_df = source_df[
    source_df["fold"] == SMOKE_TEST_FOLD
].copy()


X_train = train_df["model_text"]
y_train = train_df["target"].astype(int)

X_test = test_df["model_text"]
y_test = test_df["target"].astype(int)


print("Smoke-test fold:", SMOKE_TEST_FOLD)
print("Training sources:", len(train_df))
print("Held-out sources:", len(test_df))

print(
    "Training positives:",
    int(y_train.sum()),
)

print(
    "Held-out positives:",
    int(y_test.sum()),
)


# Confirm the expected source-level class structure.
assert len(train_df) + len(test_df) == 100

assert set(
    train_df["source_index"]
).isdisjoint(
    set(test_df["source_index"])
)

assert int(y_train.sum()) == 4
assert int(y_test.sum()) == 2


# Fit only on clean training responses.
smoke_model = build_baseline_model()

smoke_model.fit(
    X_train,
    y_train,
)


# Transform/predict held-out sources.
smoke_predictions = smoke_model.predict(
    X_test
)

smoke_scores = smoke_model.decision_function(
    X_test
)


assert len(smoke_predictions) == len(test_df)
assert len(smoke_scores) == len(test_df)

assert set(
    np.unique(smoke_predictions)
).issubset({0, 1})


# Inspect fitted TF-IDF dimensionality.
feature_count = len(
    smoke_model
    .named_steps["tfidf"]
    .get_feature_names_out()
)

print("\nSingle-fold model fit passed.")
print("TF-IDF features:", feature_count)

print(
    "Predicted positives:",
    int(smoke_predictions.sum()),
)

print(
    "Predicted negatives:",
    int(
        len(smoke_predictions)
        - smoke_predictions.sum()
    ),
)

In [ ]:
# ------------------------------------------------------------
# Full 3-fold clean-baseline cross-validation
# ------------------------------------------------------------

fold_models = {}
clean_oof_records = []

for fold_id in range(N_SPLITS):

    fold_train_df = source_df[
        source_df["fold"] != fold_id
    ].copy()

    fold_test_df = source_df[
        source_df["fold"] == fold_id
    ].copy()

    X_train = fold_train_df["model_text"]
    y_train = fold_train_df["target"].astype(int)

    X_test = fold_test_df["model_text"]
    y_test = fold_test_df["target"].astype(int)

    # Safety checks.
    assert int(y_train.sum()) == 4
    assert int(y_test.sum()) == 2

    assert set(
        fold_train_df["source_index"]
    ).isdisjoint(
        set(fold_test_df["source_index"])
    )

    # Build and fit a fresh model for this fold.
    model = build_baseline_model()

    model.fit(
        X_train,
        y_train,
    )

    fold_models[fold_id] = model

    predictions = model.predict(
        X_test
    )

    decision_scores = model.decision_function(
        X_test
    )

    for (
        (_, row),
        prediction,
        decision_score,
    ) in zip(
        fold_test_df.iterrows(),
        predictions,
        decision_scores,
    ):
        clean_oof_records.append(
            {
                "source_index": row["source_index"],
                "pair_index": row["pair_index"],
                "fold": fold_id,
                "target": int(row["target"]),
                "prediction": int(prediction),
                "decision_score": float(decision_score),
            }
        )

    print(
        f"Fold {fold_id}: "
        f"train={len(fold_train_df)}, "
        f"test={len(fold_test_df)}, "
        f"test_positive={int(y_test.sum())}, "
        f"predicted_positive={int(predictions.sum())}"
    )


clean_oof_df = pd.DataFrame(
    clean_oof_records
).sort_values(
    [
        "fold",
        "source_index",
        "pair_index",
    ]
).reset_index(drop=True)


print("\nTotal out-of-fold predictions:", len(clean_oof_df))
print(
    "Unique evaluated sources:",
    clean_oof_df["source_index"].nunique(),
)
print(
    "Actual positives:",
    int(clean_oof_df["target"].sum()),
)
print(
    "Predicted positives:",
    int(clean_oof_df["prediction"].sum()),
)

In [ ]:
# ------------------------------------------------------------
# Clean out-of-fold prediction integrity checks
# ------------------------------------------------------------

assert len(clean_oof_df) == 100

assert (
    clean_oof_df["source_index"].nunique()
    == 100
)

assert (
    clean_oof_df[
        ["source_index", "pair_index"]
    ]
    .duplicated()
    .sum()
    == 0
)

assert set(
    clean_oof_df["fold"].unique()
) == {0, 1, 2}

assert int(
    clean_oof_df["target"].sum()
) == 6

assert (
    clean_oof_df["prediction"]
    .isin([0, 1])
    .all()
)

assert (
    clean_oof_df["decision_score"]
    .notna()
    .all()
)

assert len(fold_models) == 3

print(
    "Clean out-of-fold prediction integrity checks passed."
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)


# ------------------------------------------------------------
# Clean baseline evaluation
# ------------------------------------------------------------

y_true = clean_oof_df["target"].astype(int)
y_pred = clean_oof_df["prediction"].astype(int)


accuracy = accuracy_score(
    y_true,
    y_pred,
)

balanced_accuracy = balanced_accuracy_score(
    y_true,
    y_pred,
)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0,
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0,
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0,
)

tn, fp, fn, tp = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1],
).ravel()


clean_baseline_metrics = pd.DataFrame(
    [
        {
            "accuracy": accuracy,
            "balanced_accuracy": balanced_accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        }
    ]
)


print("Clean baseline metrics:")
clean_baseline_metrics.round(3)

In [ ]:
# ------------------------------------------------------------
# Metric integrity checks
# ------------------------------------------------------------

assert tn + fp + fn + tp == 100

assert tn == 94
assert fp == 0
assert fn == 6
assert tp == 0

assert precision == 0.0
assert recall == 0.0
assert f1 == 0.0

print("Clean baseline metric checks passed.")

In [ ]:
# ------------------------------------------------------------
# Decision-score diagnostics
# ------------------------------------------------------------

score_summary = (
    clean_oof_df
    .groupby(
        [
            "fold",
            "target",
        ]
    )
    .agg(
        examples=("decision_score", "size"),
        mean_score=("decision_score", "mean"),
        min_score=("decision_score", "min"),
        max_score=("decision_score", "max"),
    )
    .round(3)
)

print("Decision-score summary by fold and class:")
score_summary

In [ ]:
# ------------------------------------------------------------
# Rank the six true positives within their held-out folds
# ------------------------------------------------------------

ranked_oof_df = clean_oof_df.copy()

ranked_oof_df["score_rank_in_fold"] = (
    ranked_oof_df
    .groupby("fold")["decision_score"]
    .rank(
        method="min",
        ascending=False,
    )
)

ranked_oof_df["score_percentile_in_fold"] = (
    ranked_oof_df
    .groupby("fold")["decision_score"]
    .rank(
        pct=True,
        ascending=True,
    )
    * 100
)

positive_score_ranks = (
    ranked_oof_df[
        ranked_oof_df["target"] == 1
    ][
        [
            "source_index",
            "pair_index",
            "fold",
            "decision_score",
            "score_rank_in_fold",
            "score_percentile_in_fold",
        ]
    ]
    .sort_values(
        [
            "fold",
            "score_rank_in_fold",
        ]
    )
    .reset_index(drop=True)
)

positive_score_ranks[
    "decision_score"
] = positive_score_ranks[
    "decision_score"
].round(3)

positive_score_ranks[
    "score_percentile_in_fold"
] = positive_score_ranks[
    "score_percentile_in_fold"
].round(1)


print("True-positive score ranks:")
positive_score_ranks

In [ ]:
# ------------------------------------------------------------
# Full robustness evaluation across all 32 conditions
# ------------------------------------------------------------

robustness_records = []

for fold_id in range(N_SPLITS):

    model = fold_models[fold_id]

    fold_eval_df = model_matrix_df[
        model_matrix_df["fold"] == fold_id
    ].copy()

    # Each held-out source should contribute all 32 variants.
    expected_sources = (
        source_df[
            source_df["fold"] == fold_id
        ][
            ["source_index", "pair_index"]
        ]
        .drop_duplicates()
        .shape[0]
    )

    assert len(fold_eval_df) == expected_sources * 32

    for (
        noise_condition,
        preprocess_config,
    ), condition_df in fold_eval_df.groupby(
        [
            "noise_condition",
            "preprocess_config",
        ],
        sort=False,
    ):

        X_eval = condition_df[
            "assistant_text_processed"
        ]

        predictions = model.predict(
            X_eval
        )

        decision_scores = model.decision_function(
            X_eval
        )

        for (
            (_, row),
            prediction,
            decision_score,
        ) in zip(
            condition_df.iterrows(),
            predictions,
            decision_scores,
        ):
            robustness_records.append(
                {
                    "source_index": row["source_index"],
                    "pair_index": row["pair_index"],
                    "fold": fold_id,
                    "target": int(row["weak_label_any"]),
                    "noise_condition": noise_condition,
                    "preprocess_config": preprocess_config,
                    "prediction": int(prediction),
                    "decision_score": float(decision_score),
                }
            )


robustness_oof_df = pd.DataFrame(
    robustness_records
)

print(
    "Robustness prediction rows:",
    len(robustness_oof_df),
)

print(
    "Unique sources:",
    robustness_oof_df["source_index"].nunique(),
)

print(
    "Experimental conditions:",
    robustness_oof_df[
        [
            "noise_condition",
            "preprocess_config",
        ]
    ]
    .drop_duplicates()
    .shape[0],
)

In [ ]:
# ------------------------------------------------------------
# Robustness prediction integrity checks
# ------------------------------------------------------------

assert len(robustness_oof_df) == 3200

assert (
    robustness_oof_df["source_index"].nunique()
    == 100
)

assert (
    robustness_oof_df[
        [
            "noise_condition",
            "preprocess_config",
        ]
    ]
    .drop_duplicates()
    .shape[0]
    == 32
)


# Each source must have exactly 32 out-of-fold predictions.
predictions_per_source = (
    robustness_oof_df
    .groupby(
        [
            "source_index",
            "pair_index",
        ]
    )
    .size()
)

assert (
    predictions_per_source == 32
).all()


# Every experimental condition must contain all 100 held-out sources.
sources_per_condition = (
    robustness_oof_df
    .groupby(
        [
            "noise_condition",
            "preprocess_config",
        ]
    )
    .size()
)

assert (
    sources_per_condition == 100
).all()


# Frozen labels remain identical across variants.
target_values_per_source = (
    robustness_oof_df
    .groupby(
        [
            "source_index",
            "pair_index",
        ]
    )["target"]
    .nunique()
)

assert (
    target_values_per_source == 1
).all()


assert (
    robustness_oof_df["prediction"]
    .isin([0, 1])
    .all()
)

assert (
    robustness_oof_df["decision_score"]
    .notna()
    .all()
)


print(
    "Full robustness prediction integrity checks passed."
)

In [ ]:
# ------------------------------------------------------------
# Classification metrics across all 32 experimental conditions
# ------------------------------------------------------------

metric_records = []

for (
    noise_condition,
    preprocess_config,
), condition_df in robustness_oof_df.groupby(
    [
        "noise_condition",
        "preprocess_config",
    ]
):

    y_true = condition_df["target"].astype(int)
    y_pred = condition_df["prediction"].astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    metric_records.append(
        {
            "noise_condition": noise_condition,
            "preprocess_config": preprocess_config,
            "accuracy": accuracy_score(
                y_true,
                y_pred,
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_true,
                y_pred,
            ),
            "precision": precision_score(
                y_true,
                y_pred,
                zero_division=0,
            ),
            "recall": recall_score(
                y_true,
                y_pred,
                zero_division=0,
            ),
            "f1": f1_score(
                y_true,
                y_pred,
                zero_division=0,
            ),
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
            "predicted_positive": int(
                y_pred.sum()
            ),
        }
    )


robustness_metrics_df = pd.DataFrame(
    metric_records
)


# Preserve experimental ordering.
noise_order = [
    "clean",
    "casing",
    "punctuation",
    "whitespace",
    "typo",
    "word_deletion",
    "filler",
    "mixed",
]

preprocess_order = [
    "none",
    "minimal",
    "noise_aware",
    "aggressive",
]

robustness_metrics_df["noise_condition"] = pd.Categorical(
    robustness_metrics_df["noise_condition"],
    categories=noise_order,
    ordered=True,
)

robustness_metrics_df["preprocess_config"] = pd.Categorical(
    robustness_metrics_df["preprocess_config"],
    categories=preprocess_order,
    ordered=True,
)

robustness_metrics_df = (
    robustness_metrics_df
    .sort_values(
        [
            "noise_condition",
            "preprocess_config",
        ]
    )
    .reset_index(drop=True)
)


display_columns = [
    "noise_condition",
    "preprocess_config",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "tn",
    "fp",
    "fn",
    "tp",
    "predicted_positive",
]

print("Classification metrics by experimental condition:")

robustness_metrics_df[
    display_columns
].round(3)

In [ ]:
# ------------------------------------------------------------
# Metric integrity checks
# ------------------------------------------------------------

assert len(robustness_metrics_df) == 32


# Every condition evaluates the same 100 sources:
# 94 negative + 6 positive.
assert (
    robustness_metrics_df[
        ["tn", "fp", "fn", "tp"]
    ]
    .sum(axis=1)
    .eq(100)
    .all()
)

assert (
    (
        robustness_metrics_df["tp"]
        + robustness_metrics_df["fn"]
    )
    == 6
).all()

assert (
    (
        robustness_metrics_df["tn"]
        + robustness_metrics_df["fp"]
    )
    == 94
).all()


# Clean + none must exactly reproduce the clean OOF baseline.
baseline_row = robustness_metrics_df[
    (
        robustness_metrics_df["noise_condition"]
        == "clean"
    )
    &
    (
        robustness_metrics_df["preprocess_config"]
        == "none"
    )
].iloc[0]

assert baseline_row["tn"] == 94
assert baseline_row["fp"] == 0
assert baseline_row["fn"] == 6
assert baseline_row["tp"] == 0

assert baseline_row["precision"] == 0.0
assert baseline_row["recall"] == 0.0
assert baseline_row["f1"] == 0.0


print(
    "32-condition classification metric checks passed."
)

In [ ]:
# ------------------------------------------------------------
# Decision-score shifts relative to clean + none baseline
# ------------------------------------------------------------

baseline_scores = (
    robustness_oof_df[
        (robustness_oof_df["noise_condition"] == "clean")
        & (robustness_oof_df["preprocess_config"] == "none")
    ][
        [
            "source_index",
            "pair_index",
            "decision_score",
        ]
    ]
    .rename(
        columns={
            "decision_score": "baseline_decision_score"
        }
    )
)

assert len(baseline_scores) == 100


score_shift_df = robustness_oof_df.merge(
    baseline_scores,
    on=[
        "source_index",
        "pair_index",
    ],
    how="left",
    validate="many_to_one",
)


score_shift_df["score_delta"] = (
    score_shift_df["decision_score"]
    - score_shift_df["baseline_decision_score"]
)

score_shift_df["absolute_score_delta"] = (
    score_shift_df["score_delta"].abs()
)


assert (
    score_shift_df["baseline_decision_score"]
    .notna()
    .all()
)


# Clean + none must have exactly zero shift.
baseline_shift_rows = score_shift_df[
    (score_shift_df["noise_condition"] == "clean")
    & (score_shift_df["preprocess_config"] == "none")
]

assert np.allclose(
    baseline_shift_rows["score_delta"],
    0.0,
)


print("Matched baseline score-shift dataset created.")
print("Rows:", len(score_shift_df))

In [ ]:
# ------------------------------------------------------------
# Summarise continuous decision-score behaviour
# ------------------------------------------------------------

score_shift_summary = (
    score_shift_df
    .groupby(
        [
            "noise_condition",
            "preprocess_config",
        ]
    )
    .agg(
        mean_score=(
            "decision_score",
            "mean",
        ),
        mean_score_positive=(
            "decision_score",
            lambda values: values[
                score_shift_df.loc[
                    values.index,
                    "target",
                ].eq(1)
            ].mean(),
        ),
        mean_score_negative=(
            "decision_score",
            lambda values: values[
                score_shift_df.loc[
                    values.index,
                    "target",
                ].eq(0)
            ].mean(),
        ),
        mean_delta=(
            "score_delta",
            "mean",
        ),
        mean_delta_positive=(
            "score_delta",
            lambda values: values[
                score_shift_df.loc[
                    values.index,
                    "target",
                ].eq(1)
            ].mean(),
        ),
        mean_delta_negative=(
            "score_delta",
            lambda values: values[
                score_shift_df.loc[
                    values.index,
                    "target",
                ].eq(0)
            ].mean(),
        ),
        mean_absolute_delta=(
            "absolute_score_delta",
            "mean",
        ),
    )
    .reset_index()
)


score_shift_summary["noise_condition"] = pd.Categorical(
    score_shift_summary["noise_condition"],
    categories=noise_order,
    ordered=True,
)

score_shift_summary["preprocess_config"] = pd.Categorical(
    score_shift_summary["preprocess_config"],
    categories=preprocess_order,
    ordered=True,
)

score_shift_summary = (
    score_shift_summary
    .sort_values(
        [
            "noise_condition",
            "preprocess_config",
        ]
    )
    .reset_index(drop=True)
)


score_columns = [
    "mean_score",
    "mean_score_positive",
    "mean_score_negative",
    "mean_delta",
    "mean_delta_positive",
    "mean_delta_negative",
    "mean_absolute_delta",
]

score_shift_summary[
    score_columns
] = score_shift_summary[
    score_columns
].round(4)


print("Decision-score shifts by experimental condition:")
score_shift_summary

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
)


# ------------------------------------------------------------
# Fold-wise ranking metrics for all 32 conditions
# ------------------------------------------------------------

ranking_metric_records = []

for (
    noise_condition,
    preprocess_config,
    fold_id,
), fold_condition_df in robustness_oof_df.groupby(
    [
        "noise_condition",
        "preprocess_config",
        "fold",
    ]
):

    y_true = (
        fold_condition_df["target"]
        .astype(int)
    )

    scores = (
        fold_condition_df["decision_score"]
        .astype(float)
    )

    # Each held-out fold must contain both classes.
    assert y_true.nunique() == 2
    assert int(y_true.sum()) == 2

    ranking_metric_records.append(
        {
            "noise_condition": noise_condition,
            "preprocess_config": preprocess_config,
            "fold": int(fold_id),
            "roc_auc": roc_auc_score(
                y_true,
                scores,
            ),
            "average_precision": average_precision_score(
                y_true,
                scores,
            ),
        }
    )


fold_ranking_metrics_df = pd.DataFrame(
    ranking_metric_records
)

assert len(fold_ranking_metrics_df) == 96

print(
    "Fold-condition ranking metric rows:",
    len(fold_ranking_metrics_df),
)

In [ ]:
# ------------------------------------------------------------
# Macro-average ranking metrics across the three folds
# ------------------------------------------------------------

ranking_summary_df = (
    fold_ranking_metrics_df
    .groupby(
        [
            "noise_condition",
            "preprocess_config",
        ]
    )
    .agg(
        mean_roc_auc=(
            "roc_auc",
            "mean",
        ),
        min_roc_auc=(
            "roc_auc",
            "min",
        ),
        max_roc_auc=(
            "roc_auc",
            "max",
        ),
        mean_average_precision=(
            "average_precision",
            "mean",
        ),
    )
    .reset_index()
)


ranking_summary_df["noise_condition"] = pd.Categorical(
    ranking_summary_df["noise_condition"],
    categories=noise_order,
    ordered=True,
)

ranking_summary_df["preprocess_config"] = pd.Categorical(
    ranking_summary_df["preprocess_config"],
    categories=preprocess_order,
    ordered=True,
)

ranking_summary_df = (
    ranking_summary_df
    .sort_values(
        [
            "noise_condition",
            "preprocess_config",
        ]
    )
    .reset_index(drop=True)
)


for column in [
    "mean_roc_auc",
    "min_roc_auc",
    "max_roc_auc",
    "mean_average_precision",
]:
    ranking_summary_df[column] = (
        ranking_summary_df[column]
        .round(3)
    )


print("Fold-macro ranking metrics:")
ranking_summary_df

In [ ]:
# ------------------------------------------------------------
# Ranking-metric deltas and preprocessing recovery
# ------------------------------------------------------------

baseline_ranking_row = ranking_summary_df[
    (ranking_summary_df["noise_condition"] == "clean")
    & (ranking_summary_df["preprocess_config"] == "none")
].iloc[0]

baseline_roc_auc = float(
    baseline_ranking_row["mean_roc_auc"]
)

baseline_average_precision = float(
    baseline_ranking_row["mean_average_precision"]
)


ranking_delta_df = ranking_summary_df.copy()


# Change relative to the clean + none baseline.
ranking_delta_df["roc_auc_delta_vs_clean"] = (
    ranking_delta_df["mean_roc_auc"]
    - baseline_roc_auc
)

ranking_delta_df["ap_delta_vs_clean"] = (
    ranking_delta_df["mean_average_precision"]
    - baseline_average_precision
)


# Obtain the unprocessed ('none') score for each noise condition.
noise_none_reference = (
    ranking_delta_df[
        ranking_delta_df["preprocess_config"] == "none"
    ][
        [
            "noise_condition",
            "mean_roc_auc",
            "mean_average_precision",
        ]
    ]
    .rename(
        columns={
            "mean_roc_auc": "noise_none_roc_auc",
            "mean_average_precision": "noise_none_average_precision",
        }
    )
)


ranking_delta_df = ranking_delta_df.merge(
    noise_none_reference,
    on="noise_condition",
    how="left",
    validate="many_to_one",
)


# Positive recovery means preprocessing improved ranking relative
# to leaving that same noisy text unprocessed.
ranking_delta_df["roc_auc_recovery_vs_none"] = (
    ranking_delta_df["mean_roc_auc"]
    - ranking_delta_df["noise_none_roc_auc"]
)

ranking_delta_df["ap_recovery_vs_none"] = (
    ranking_delta_df["mean_average_precision"]
    - ranking_delta_df["noise_none_average_precision"]
)


delta_columns = [
    "roc_auc_delta_vs_clean",
    "ap_delta_vs_clean",
    "roc_auc_recovery_vs_none",
    "ap_recovery_vs_none",
]

ranking_delta_df[
    delta_columns
] = ranking_delta_df[
    delta_columns
].round(3)


print("Ranking degradation and preprocessing recovery:")
ranking_delta_df[
    [
        "noise_condition",
        "preprocess_config",
        "mean_roc_auc",
        "mean_average_precision",
        "roc_auc_delta_vs_clean",
        "ap_delta_vs_clean",
        "roc_auc_recovery_vs_none",
        "ap_recovery_vs_none",
    ]
]

In [ ]:
# ------------------------------------------------------------
# Individual positive-source robustness diagnostics
# ------------------------------------------------------------

positive_shift_df = score_shift_df[
    score_shift_df["target"] == 1
].copy()


# Exclude the clean + none reference row when searching for
# the largest experimental movements.
positive_experimental_df = positive_shift_df[
    ~(
        (positive_shift_df["noise_condition"] == "clean")
        & (positive_shift_df["preprocess_config"] == "none")
    )
].copy()


positive_source_records = []

for (
    source_index,
    pair_index,
), source_rows in positive_experimental_df.groupby(
    [
        "source_index",
        "pair_index",
    ]
):

    baseline_score = float(
        source_rows["baseline_decision_score"].iloc[0]
    )

    worst_idx = (
        source_rows["score_delta"]
        .idxmin()
    )

    best_idx = (
        source_rows["score_delta"]
        .idxmax()
    )

    worst_row = source_rows.loc[worst_idx]
    best_row = source_rows.loc[best_idx]

    positive_source_records.append(
        {
            "source_index": source_index,
            "pair_index": pair_index,
            "fold": int(source_rows["fold"].iloc[0]),
            "baseline_score": baseline_score,
            "worst_score_delta": float(
                worst_row["score_delta"]
            ),
            "worst_noise": worst_row[
                "noise_condition"
            ],
            "worst_preprocess": worst_row[
                "preprocess_config"
            ],
            "best_score_delta": float(
                best_row["score_delta"]
            ),
            "best_noise": best_row[
                "noise_condition"
            ],
            "best_preprocess": best_row[
                "preprocess_config"
            ],
            "mean_absolute_delta": float(
                source_rows[
                    "absolute_score_delta"
                ].mean()
            ),
        }
    )


positive_source_stability_df = pd.DataFrame(
    positive_source_records
).sort_values(
    "worst_score_delta"
).reset_index(drop=True)


for column in [
    "baseline_score",
    "worst_score_delta",
    "best_score_delta",
    "mean_absolute_delta",
]:
    positive_source_stability_df[column] = (
        positive_source_stability_df[column]
        .round(4)
    )


assert len(positive_source_stability_df) == 6


print(
    "Positive-source decision-score stability:"
)

positive_source_stability_df

In [ ]:
# ------------------------------------------------------------
# Positive-class score sensitivity by preprocessing strategy
# ------------------------------------------------------------

positive_preprocess_sensitivity = (
    positive_shift_df
    .groupby("preprocess_config")
    .agg(
        examples=("score_delta", "size"),
        mean_delta=("score_delta", "mean"),
        mean_absolute_delta=(
            "absolute_score_delta",
            "mean",
        ),
        min_delta=("score_delta", "min"),
        max_delta=("score_delta", "max"),
    )
    .reindex(preprocess_order)
)


positive_preprocess_sensitivity[
    [
        "mean_delta",
        "mean_absolute_delta",
        "min_delta",
        "max_delta",
    ]
] = (
    positive_preprocess_sensitivity[
        [
            "mean_delta",
            "mean_absolute_delta",
            "min_delta",
            "max_delta",
        ]
    ]
    .round(4)
)


print(
    "Positive-class sensitivity by preprocessing configuration:"
)

positive_preprocess_sensitivity

In [ ]:
# ------------------------------------------------------------
# Compare aggressive vs non-aggressive perturbation magnitude
# ------------------------------------------------------------

aggressive_positive_rows = positive_shift_df[
    positive_shift_df["preprocess_config"]
    == "aggressive"
]

non_aggressive_positive_rows = positive_shift_df[
    positive_shift_df["preprocess_config"]
    != "aggressive"
]


aggressive_mean_abs = (
    aggressive_positive_rows[
        "absolute_score_delta"
    ].mean()
)

non_aggressive_mean_abs = (
    non_aggressive_positive_rows[
        "absolute_score_delta"
    ].mean()
)


print(
    "Mean absolute positive-score shift — aggressive:",
    round(aggressive_mean_abs, 4),
)

print(
    "Mean absolute positive-score shift — non-aggressive:",
    round(non_aggressive_mean_abs, 4),
)

print(
    "Aggressive / non-aggressive shift ratio:",
    round(
        aggressive_mean_abs
        / non_aggressive_mean_abs,
        2,
    ),
)

## Pilot modelling findings

The fixed TF-IDF + linear SVM baseline was evaluated using three-fold
stratified cross-validation at the source-conversation level.

All derived variants of a source conversation remained in the same held-out
fold, preventing leakage between training and evaluation data.

### Default-threshold classification

Across the 100 source conversations, the clean baseline produced:

- accuracy: `0.94`;
- balanced accuracy: `0.50`;
- precision: `0.00`;
- recall: `0.00`;
- F1: `0.00`;
- true negatives: `94`;
- false positives: `0`;
- false negatives: `6`;
- true positives: `0`.

The same hard-prediction outcome occurred across all 32 experimental
noise/preprocessing combinations.

The high ordinary accuracy therefore reflects class imbalance rather than
successful detection of the positive class.

### Decision-score behaviour

Although no positive example crossed the default SVM decision threshold, the
continuous decision scores showed useful relative discrimination.

For the clean baseline:

- mean fold-level ROC-AUC: `0.867`;
- mean fold-level Average Precision: `0.537`.

Several positive examples ranked near the top of their held-out folds despite
remaining below the classification threshold.

### Noise robustness

Casing and punctuation produced effectively no change in the model output.
This is consistent with lowercasing in the TF-IDF representation and the
word-based tokenization used by the baseline.

Whitespace, typo, word deletion, filler, and mixed noise produced relatively
small changes in decision scores. Small apparent improvements under some noisy
conditions are treated cautiously because the pilot contains only six positive
source examples.

Neither `minimal` nor `noise_aware` preprocessing produced consistent ranking
recovery relative to leaving the corresponding noisy text unprocessed.

### Aggressive preprocessing

Aggressive preprocessing produced the largest and most consistent degradation.

Across the positive pilot examples:

- mean absolute score shift with aggressive preprocessing: `0.1607`;
- mean absolute score shift across non-aggressive preprocessing: `0.0092`;
- aggressive/non-aggressive shift ratio: approximately `17.39`.

The aggressive configuration also reduced ranking performance across the
experimental conditions. For example:

- clean ROC-AUC decreased from `0.867` to `0.755`;
- clean Average Precision decreased from `0.537` to `0.467`;
- mixed ROC-AUC decreased from `0.863` to `0.782`;
- mixed Average Precision decreased from `0.551` to `0.473`.

This behaviour is consistent with the cue-preservation diagnostics from
Notebook 04, where aggressive preprocessing removed function words and short
linguistic constructions associated with the frozen directive-advice cues.

These findings are descriptive pilot diagnostics. The repeated experimental
variants are derived from only 100 source conversations and six positive
sources, so the results should not be interpreted as statistically stable
estimates of generalisable classifier performance.

In [ ]:
# ------------------------------------------------------------
# Figure 1 — ROC-AUC robustness across experimental conditions
# ------------------------------------------------------------

import matplotlib.pyplot as plt


FIGURES_DIR = (
    PROJECT_ROOT
    / "results"
    / "figures"
)

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


roc_plot_df = (
    ranking_summary_df
    .pivot(
        index="noise_condition",
        columns="preprocess_config",
        values="mean_roc_auc",
    )
    .reindex(noise_order)
    .reindex(
        columns=preprocess_order
    )
)


plt.figure(
    figsize=(11, 6)
)

for config in preprocess_order:
    plt.plot(
        roc_plot_df.index,
        roc_plot_df[config],
        marker="o",
        label=config,
    )


plt.axhline(
    y=0.5,
    linestyle="--",
    linewidth=1,
    label="chance level",
)

plt.xlabel(
    "Noise condition"
)

plt.ylabel(
    "Mean fold ROC-AUC"
)

plt.title(
    "Pilot SVM Robustness Across Noise and Preprocessing Conditions"
)

plt.xticks(
    rotation=35,
    ha="right",
)

plt.ylim(
    0.45,
    1.00,
)

plt.legend(
    title="Preprocessing"
)

plt.tight_layout()


roc_figure_path = (
    FIGURES_DIR
    / "pilot_roc_auc_robustness.png"
)

plt.savefig(
    roc_figure_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()


print("Saved ROC-AUC figure:")
print(roc_figure_path)
print(
    "File exists:",
    roc_figure_path.exists(),
)

In [ ]:
# ------------------------------------------------------------
# Figure 2 — Average Precision robustness across conditions
# ------------------------------------------------------------

ap_plot_df = (
    ranking_summary_df
    .pivot(
        index="noise_condition",
        columns="preprocess_config",
        values="mean_average_precision",
    )
    .reindex(noise_order)
    .reindex(
        columns=preprocess_order
    )
)


plt.figure(
    figsize=(11, 6)
)

for config in preprocess_order:
    plt.plot(
        ap_plot_df.index,
        ap_plot_df[config],
        marker="o",
        label=config,
    )


# Positive-class prevalence in the pilot.
pilot_positive_prevalence = (
    source_df["target"].mean()
)

plt.axhline(
    y=pilot_positive_prevalence,
    linestyle="--",
    linewidth=1,
    label="positive prevalence",
)


plt.xlabel(
    "Noise condition"
)

plt.ylabel(
    "Mean fold Average Precision"
)

plt.title(
    "Pilot SVM Average Precision Across Noise and Preprocessing Conditions"
)

plt.xticks(
    rotation=35,
    ha="right",
)

plt.ylim(
    0.0,
    0.65,
)

plt.legend(
    title="Preprocessing"
)

plt.tight_layout()


ap_figure_path = (
    FIGURES_DIR
    / "pilot_average_precision_robustness.png"
)

plt.savefig(
    ap_figure_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()


print("Saved Average Precision figure:")
print(ap_figure_path)

print(
    "File exists:",
    ap_figure_path.exists(),
)

print(
    "Pilot positive prevalence:",
    round(
        pilot_positive_prevalence,
        3,
    ),
)

In [ ]:
# ------------------------------------------------------------
# Figure 3 — Positive-class decision-score shifts
# ------------------------------------------------------------

positive_shift_plot_df = (
    score_shift_summary
    .pivot(
        index="noise_condition",
        columns="preprocess_config",
        values="mean_delta_positive",
    )
    .reindex(noise_order)
    .reindex(
        columns=preprocess_order
    )
)


plt.figure(
    figsize=(11, 6)
)

for config in preprocess_order:
    plt.plot(
        positive_shift_plot_df.index,
        positive_shift_plot_df[config],
        marker="o",
        label=config,
    )


# Zero represents no movement from each source's clean + none baseline.
plt.axhline(
    y=0.0,
    linestyle="--",
    linewidth=1,
    label="clean-baseline score",
)


plt.xlabel(
    "Noise condition"
)

plt.ylabel(
    "Mean decision-score shift for positive sources"
)

plt.title(
    "Effect of Noise and Preprocessing on Positive-Class SVM Scores"
)

plt.xticks(
    rotation=35,
    ha="right",
)

plt.legend(
    title="Preprocessing"
)

plt.tight_layout()


positive_shift_figure_path = (
    FIGURES_DIR
    / "pilot_positive_score_shift.png"
)

plt.savefig(
    positive_shift_figure_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()


print("Saved positive-score shift figure:")
print(positive_shift_figure_path)

print(
    "File exists:",
    positive_shift_figure_path.exists(),
)

In [ ]:
# ------------------------------------------------------------
# Export pilot modelling results
# ------------------------------------------------------------

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "tables"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# 1. Source-level fold assignments
fold_assignments_path = (
    RESULTS_DIR
    / "pilot_source_fold_assignments.csv"
)

source_df[
    [
        "source_index",
        "pair_index",
        "target",
        "fold",
    ]
].to_csv(
    fold_assignments_path,
    index=False,
)


# 2. Clean out-of-fold predictions
clean_oof_path = (
    RESULTS_DIR
    / "pilot_clean_oof_predictions.csv"
)

clean_oof_df.to_csv(
    clean_oof_path,
    index=False,
)


# 3. Full 3,200-row robustness predictions
robustness_predictions_path = (
    RESULTS_DIR
    / "pilot_robustness_oof_predictions.csv"
)

robustness_oof_df.to_csv(
    robustness_predictions_path,
    index=False,
)


# 4. Threshold-based metrics
classification_metrics_path = (
    RESULTS_DIR
    / "pilot_classification_metrics.csv"
)

robustness_metrics_df.to_csv(
    classification_metrics_path,
    index=False,
)


# 5. Fold-level ranking metrics
fold_ranking_path = (
    RESULTS_DIR
    / "pilot_fold_ranking_metrics.csv"
)

fold_ranking_metrics_df.to_csv(
    fold_ranking_path,
    index=False,
)


# 6. Macro ranking summary
ranking_summary_path = (
    RESULTS_DIR
    / "pilot_ranking_summary.csv"
)

ranking_summary_df.to_csv(
    ranking_summary_path,
    index=False,
)


# 7. Ranking degradation / recovery
ranking_delta_path = (
    RESULTS_DIR
    / "pilot_ranking_deltas.csv"
)

ranking_delta_df.to_csv(
    ranking_delta_path,
    index=False,
)


# 8. Decision-score shift summary
score_shift_summary_path = (
    RESULTS_DIR
    / "pilot_score_shift_summary.csv"
)

score_shift_summary.to_csv(
    score_shift_summary_path,
    index=False,
)


# 9. Positive-source stability
positive_stability_path = (
    RESULTS_DIR
    / "pilot_positive_source_stability.csv"
)

positive_source_stability_df.to_csv(
    positive_stability_path,
    index=False,
)


exported_paths = [
    fold_assignments_path,
    clean_oof_path,
    robustness_predictions_path,
    classification_metrics_path,
    fold_ranking_path,
    ranking_summary_path,
    ranking_delta_path,
    score_shift_summary_path,
    positive_stability_path,
]


print("Exported modelling result files:")

for path in exported_paths:
    print(
        f"- {path.name}:",
        path.exists(),
    )

In [ ]:
# ------------------------------------------------------------
# Verify exported result tables
# ------------------------------------------------------------

assert len(
    pd.read_csv(fold_assignments_path)
) == 100

assert len(
    pd.read_csv(clean_oof_path)
) == 100

assert len(
    pd.read_csv(robustness_predictions_path)
) == 3200

assert len(
    pd.read_csv(classification_metrics_path)
) == 32

assert len(
    pd.read_csv(fold_ranking_path)
) == 96

assert len(
    pd.read_csv(ranking_summary_path)
) == 32

assert len(
    pd.read_csv(ranking_delta_path)
) == 32

assert len(
    pd.read_csv(score_shift_summary_path)
) == 32

assert len(
    pd.read_csv(positive_stability_path)
) == 6


print("Exported modelling results verification passed.")

## Final pilot modelling outcome

The pilot modelling and robustness-evaluation stage is complete.

### Data and evaluation design

The experiment used:

- 100 unique source conversations;
- 6 weak-label-positive sources;
- 94 weak-label-negative sources;
- 3-fold stratified cross-validation;
- source-level fold assignment to prevent leakage;
- clean + none text only for model fitting;
- all 32 noise/preprocessing combinations for held-out robustness evaluation.

Each source conversation was evaluated out-of-fold exactly once under every
experimental condition.

### Baseline model

The fixed baseline consisted of:

- TF-IDF word unigrams and bigrams;
- lowercase normalization;
- sublinear term-frequency scaling;
- L2 normalization;
- linear SVM (`LinearSVC`);
- `C=1.0`;
- balanced class weighting.

No hyperparameter tuning was performed.

### Default-threshold results

For the clean baseline:

- accuracy: `0.94`;
- balanced accuracy: `0.50`;
- precision: `0.00`;
- recall: `0.00`;
- F1: `0.00`;
- TN: `94`;
- FP: `0`;
- FN: `6`;
- TP: `0`.

The same hard-prediction outcome occurred across all 32 experimental
conditions.

Ordinary accuracy is therefore misleading in this highly imbalanced pilot.

### Ranking behaviour

Despite the absence of positive hard predictions, the continuous SVM decision
scores showed useful ranking behaviour.

For the clean baseline:

- mean fold ROC-AUC: `0.867`;
- mean fold Average Precision: `0.537`;
- pilot positive prevalence: `0.06`.

Several positive examples ranked near the top of their held-out folds even
though their scores remained below the default SVM decision threshold.

### Robustness findings

Casing and punctuation produced effectively no measurable change in model
ranking.

Whitespace, typo, word deletion, filler, and mixed noise produced relatively
small score changes. Small apparent improvements under some noisy conditions
are treated as pilot instability rather than evidence that noise improves
classification.

`minimal` and `noise_aware` preprocessing did not provide consistent ranking
recovery relative to leaving the corresponding noisy text unprocessed.

Aggressive preprocessing produced the clearest degradation:

- clean ROC-AUC: `0.867` → `0.755`;
- clean Average Precision: `0.537` → `0.467`;
- mixed ROC-AUC: `0.863` → `0.782`;
- mixed Average Precision: `0.551` → `0.473`.

Across positive-source variants:

- aggressive mean absolute decision-score shift: `0.1607`;
- non-aggressive mean absolute decision-score shift: `0.0092`;
- aggressive/non-aggressive shift ratio: approximately `17.39`.

Aggressive preprocessing consistently moved positive examples farther from the
positive SVM decision boundary.

### Interpretation

The pilot demonstrates that preprocessing choices can materially alter
task-relevant classifier behaviour even when hard classification metrics do
not change.

The result also illustrates a distinction between:

- preservation of exact rule-based linguistic cues; and
- preservation of broader classifier ranking behaviour.

These results remain pilot diagnostics because they are derived from only 100
source conversations and six positive examples. They should not be interpreted
as statistically stable or population-level estimates of model performance.

### Saved outputs

Three figures were generated:

- `results/figures/pilot_roc_auc_robustness.png`
- `results/figures/pilot_average_precision_robustness.png`
- `results/figures/pilot_positive_score_shift.png`

Nine machine-readable result tables were exported under `results/tables/`,
including fold assignments, out-of-fold predictions, classification metrics,
ranking metrics, ranking deltas, score-shift summaries, and positive-source
stability results.

These outputs provide the reproducible pilot evidence required to assess the
feasibility of the full robustness experiment.